# Practical Work #8 — Application of ML Algorithms and Advanced Methods for Image Analysis

**Student:** Tishkin Mykyta

**Dataset:** Fashion-MNIST (10 clothing categories, 28x28 grayscale images)
**Source:** [Zalando Research — Fashion-MNIST](https://www.kaggle.com/datasets/zalando-research/fashionmnist)

**Goal:** Build, compare, and evaluate multiple deep learning architectures (CNN, pretrained models, Vision Transformers, hybrid models) for image classification, applying advanced augmentation strategies, explainability techniques (Grad-CAM, SHAP), and class imbalance handling methods.

**Pipeline:**
1. **EDA** — class distribution, image statistics, imbalance detection
2. **Preprocessing** — resizing, normalization, augmentation, stratified split (70/15/15), 5-fold CV
3. **Baseline & Pretrained Models** — custom CNN, ResNet18/50, EfficientNet-B0, VGG16, DenseNet121
4. **Advanced Techniques** — augmentation study, Mixup/CutMix, generative augmentation, ViT, hybrid CNN-Transformer
5. **Explainability** — Grad-CAM and SHAP feature attribution
6. **Imbalance Handling** — Focal Loss, Weighted Sampling
7. **Evaluation** — Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, t-SNE, threshold analysis

> Run in Google Colab with GPU: **Runtime > Change runtime type > GPU (T4)**

## Step 1 — Install Dependencies

We install the required libraries: **PyTorch** and **torchvision** for deep learning, **timm** for Vision Transformer models, **grad-cam** and **shap** for explainability, **scikit-learn** for metrics and cross-validation, and **scikit-image** for image resizing in Grad-CAM visualizations.

In [ ]:
!pip install torch torchvision timm grad-cam shap scikit-learn scikit-image python-docx \
             pandas numpy matplotlib seaborn --quiet

## Step 2 — Imports & Configuration

Import all necessary libraries and configure the environment:
- **PyTorch** — model building, training, GPU acceleration
- **torchvision** — pretrained models (ResNet, EfficientNet, VGG, DenseNet), image transforms
- **timm** — Vision Transformer (ViT) models
- **scikit-learn** — stratified splitting, cross-validation, classification metrics (F1, AUC, confusion matrix)
- **matplotlib / seaborn** — visualization

We also set `SEED = 42` for full reproducibility across all random operations.

In [ ]:
import os, time, copy, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler

import torchvision
import torchvision.transforms as T
from torchvision import models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, precision_recall_curve,
                              average_precision_score, accuracy_score,
                              f1_score, roc_auc_score, precision_score,
                              recall_score)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

import timm

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Step 3 — Load the Fashion-MNIST Dataset

Fashion-MNIST is a dataset of Zalando's article images consisting of **70,000 grayscale images** (28x28 pixels) across **10 clothing categories**: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, and Ankle boot.

We combine the standard train (60,000) and test (10,000) splits into one pool of 70,000 images, which we will later re-split with a stratified 70/15/15 ratio to ensure balanced representation across all classes.

In [ ]:
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
NUM_CLASSES = 10

raw_train = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True)
raw_test  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)

all_images = np.concatenate([raw_train.data.numpy(), raw_test.data.numpy()], axis=0)
all_labels = np.concatenate([raw_train.targets.numpy(), raw_test.targets.numpy()], axis=0)

print(f"Total images: {len(all_images)}")
print(f"Image shape : {all_images[0].shape} (H x W)")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## Step 4 — Exploratory Data Analysis (EDA)

Before building models, we perform a comprehensive EDA to understand the dataset characteristics:

1. **Class Distribution** — verify whether classes are balanced (each should have ~7,000 samples)
2. **Mean Pixel Intensity per Class** — different clothing types have different average brightness (e.g., bags tend to be darker)
3. **Pixel Value Distribution** — check the overall distribution of pixel values across all images
4. **Sample Images** — visual inspection of one representative image per class
5. **Per-Image Mean vs. Std** — scatter plot to understand the variance in image brightness/contrast
6. **Imbalance Ratio** — quantify the ratio between the largest and smallest class to confirm balance

These analyses inform our preprocessing and modeling decisions (e.g., whether class imbalance handling is needed).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Exploratory Data Analysis  —  Fashion-MNIST", fontsize=15, fontweight='bold')

# 4a — Class distribution
counts = np.bincount(all_labels)
colors = sns.color_palette("Blues_d", NUM_CLASSES)
axes[0, 0].bar(range(NUM_CLASSES), counts, color=colors, edgecolor='white')
axes[0, 0].set_xticks(range(NUM_CLASSES))
axes[0, 0].set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=8)
axes[0, 0].set_title("Class Distribution", fontweight='bold')
axes[0, 0].set_ylabel("Count")
for i, v in enumerate(counts):
    axes[0, 0].text(i, v + 50, str(v), ha='center', fontsize=7)

# 4b — Mean pixel intensity per class
class_means = [all_images[all_labels == c].mean() for c in range(NUM_CLASSES)]
axes[0, 1].bar(range(NUM_CLASSES), class_means,
               color=sns.color_palette("viridis", NUM_CLASSES), edgecolor='white')
axes[0, 1].set_xticks(range(NUM_CLASSES))
axes[0, 1].set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=8)
axes[0, 1].set_title("Mean Pixel Intensity per Class", fontweight='bold')
axes[0, 1].set_ylabel("Mean pixel value")

# 4c — Pixel value distribution
axes[0, 2].hist(all_images.flatten(), bins=50, color='steelblue',
                edgecolor='white', density=True)
axes[0, 2].set_title("Pixel Value Distribution", fontweight='bold')
axes[0, 2].set_xlabel("Pixel Value (0-255)")
axes[0, 2].set_ylabel("Density")
axes[0, 2].axvline(all_images.mean(), color='crimson', ls='--',
                    label=f"Mean: {all_images.mean():.1f}")
axes[0, 2].legend()

# 4d — Sample images
axes[1, 0].set_title("Sample Images (1 per class)", fontweight='bold')
axes[1, 0].axis('off')
grid_img = [all_images[np.where(all_labels == i)[0][0]] for i in range(NUM_CLASSES)]
axes[1, 0].imshow(np.hstack(grid_img), cmap='gray')

# 4e — Per-image mean vs std
img_means = all_images.reshape(len(all_images), -1).mean(axis=1)
img_stds  = all_images.reshape(len(all_images), -1).std(axis=1)
axes[1, 1].scatter(img_means, img_stds, alpha=0.01, s=1, c='steelblue')
axes[1, 1].set_title("Per-Image Mean vs Std", fontweight='bold')
axes[1, 1].set_xlabel("Mean pixel"); axes[1, 1].set_ylabel("Std pixel")

# 4f — Imbalance ratio
imbalance_ratio = counts / counts.min()
axes[1, 2].bar(range(NUM_CLASSES), imbalance_ratio,
               color=sns.color_palette("RdYlGn", NUM_CLASSES), edgecolor='white')
axes[1, 2].set_xticks(range(NUM_CLASSES))
axes[1, 2].set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=8)
axes[1, 2].set_title("Imbalance Ratio (vs smallest class)", fontweight='bold')
axes[1, 2].set_ylabel("Ratio")
axes[1, 2].axhline(1.0, color='gray', ls='--', lw=0.8)

plt.tight_layout()
plt.savefig("fig1_eda.png", bbox_inches='tight')
plt.show()

print(f"\nImage resolution: 28 x 28 (grayscale)")
print(f"Pixel range: [{all_images.min()}, {all_images.max()}]")
print(f"Global mean: {all_images.mean():.2f}, std: {all_images.std():.2f}")
print(f"Class counts: {dict(zip(CLASS_NAMES, counts))}")
print(f"Imbalance ratio (max/min): {counts.max()/counts.min():.2f} — dataset is balanced")

## Step 5 — Data Preprocessing: Stratified Split, Transforms & DataLoaders

**Split strategy (70 / 15 / 15):**
- **Train (70%)** — used for model training with augmentation
- **Validation (15%)** — used for hyperparameter tuning and early stopping (best F1 checkpoint)
- **Test (15%)** — held out for final, unbiased evaluation

The split is **stratified**, preserving class proportions in each subset. This prevents any subset from being unrepresentative.

**Preprocessing pipeline:**
- **Resize** to 224x224 — required by pretrained ImageNet models (ResNet, EfficientNet, VGG, etc.)
- **Normalization** — mean and std computed from the training split only to prevent data leakage
- **Grayscale to 3-channel** — replicate the single channel to match ImageNet's 3-channel input
- **Training augmentation** — random horizontal flip, rotation (+-10 deg), and translation (10%) to improve generalization

A custom `FashionSubset` Dataset class wraps the numpy arrays with on-the-fly transforms.

In [ ]:
# Stratified 70 / 15 / 15 split
train_idx, temp_idx = train_test_split(
    np.arange(len(all_labels)), test_size=0.30,
    random_state=SEED, stratify=all_labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    random_state=SEED, stratify=all_labels[temp_idx]
)

print(f"Split sizes — Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# Normalization stats (computed on train split only — no leakage)
train_mean = all_images[train_idx].mean() / 255.0
train_std  = all_images[train_idx].std()  / 255.0
print(f"Normalization (train): mean={train_mean:.4f}, std={train_std:.4f}")

IMG_SIZE = 224

transform_train = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize([train_mean]*3, [train_std]*3),
])

transform_eval = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize([train_mean]*3, [train_std]*3),
])


class FashionSubset(Dataset):
    """Custom dataset wrapping numpy arrays with transforms."""
    def __init__(self, images, labels, indices, transform=None):
        self.images    = images[indices]
        self.labels    = labels[indices]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = FashionSubset(all_images, all_labels, train_idx, transform_train)
val_ds   = FashionSubset(all_images, all_labels, val_idx,   transform_eval)
test_ds  = FashionSubset(all_images, all_labels, test_idx,  transform_eval)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## Step 6 — 5-Fold Cross-Validation Setup

We prepare a **Stratified 5-Fold Cross-Validation** split on the training data. This is used later (Cell 15) to evaluate ResNet18's stability and obtain a more robust estimate of generalization performance.

Each fold contains approximately equal numbers of samples, and the stratification ensures that class proportions are maintained within every fold. This guards against variance in results due to lucky/unlucky splits.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(train_idx, all_labels[train_idx]))
print(f"5-Fold CV prepared — fold sizes: {[len(f[1]) for f in fold_splits]}")

## Step 7 — Training Utilities

We define three reusable functions for the entire experiment:

1. **`train_one_epoch`** — standard training loop (forward pass, loss computation, backpropagation, parameter update)
2. **`evaluate`** — inference loop that returns loss, accuracy, predictions, true labels, and class probabilities (used for all metrics)
3. **`train_model`** — full training procedure with:
   - Epoch-level logging of train/val loss and accuracy
   - **Best-model checkpointing** based on validation F1-macro score (saves the best weights)
   - Optional learning rate scheduling (ReduceLROnPlateau)
   - Elapsed time tracking for fair comparison of training costs

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        probs = F.softmax(outputs, dim=1)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (total_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs=10, model_name="Model"):
    """Full training loop with best-model checkpointing."""
    best_val_f1, best_state = 0, None
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()

    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, vl_pred, vl_true, _ = evaluate(model, val_loader, criterion, device)
        vl_f1 = f1_score(vl_true, vl_pred, average='macro')

        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(vl_loss)
            else:
                scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_state  = copy.deepcopy(model.state_dict())

        print(f"  Epoch {epoch+1:>2}/{epochs} | "
              f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
              f"Val Loss={vl_loss:.4f} Acc={vl_acc:.4f} F1={vl_f1:.4f}")

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    print(f"  {model_name} done in {elapsed:.1f}s | Best Val F1: {best_val_f1:.4f}")
    return model, history, elapsed

## Step 8 — Baseline CNN

Our baseline is a simple **3-layer CNN** built from scratch (no pretrained weights):

| Layer | Details |
|-------|---------|
| Conv Block 1 | Conv2d(3→32, 3x3) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 2 | Conv2d(32→64, 3x3) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 3 | Conv2d(64→128, 3x3) → BatchNorm → ReLU → AdaptiveAvgPool(4) |
| Classifier | Flatten → Linear(2048→256) → ReLU → Dropout(0.4) → Linear(256→10) |

This serves as a **lower bound** — we expect all pretrained models to outperform it, since they leverage features learned on ImageNet (1.2M images, 1000 classes).

**Training:** Adam optimizer (lr=1e-3), weight decay=1e-4, ReduceLROnPlateau scheduler, 10 epochs.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


print("=" * 60)
print("Training Baseline CNN...")
print("=" * 60)
baseline_cnn = BaselineCNN(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

baseline_cnn, hist_baseline, time_baseline = train_model(
    baseline_cnn, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="Baseline CNN"
)

## Step 9 — ResNet18 (Transfer Learning + Fine-tuning)

**ResNet18** (11.7M parameters) uses residual skip connections that allow training of deeper networks by mitigating the vanishing gradient problem.

We apply a **two-phase training strategy**:

- **Phase 1 — Transfer Learning (5 epochs):** Freeze the entire backbone (convolutional layers pretrained on ImageNet) and only train the newly added classification head (`fc` layer). This is fast and leverages learned feature representations.
- **Phase 2 — Fine-tuning (10 epochs):** Unfreeze all layers and train end-to-end with a lower learning rate (1e-4 vs 1e-3). This adapts the backbone features to Fashion-MNIST's specific visual patterns.

This two-phase approach avoids catastrophic forgetting of pretrained knowledge while still allowing domain-specific adaptation.

In [ ]:
print("=" * 60)
print("Training ResNet18 — Phase 1: Transfer Learning (frozen backbone)...")
print("=" * 60)

resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet18.fc = nn.Linear(resnet18.fc.in_features, NUM_CLASSES)

for param in resnet18.parameters():
    param.requires_grad = False
for param in resnet18.fc.parameters():
    param.requires_grad = True

resnet18 = resnet18.to(DEVICE)
optimizer = optim.Adam(resnet18.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

resnet18, hist_r18_tl, time_r18_tl = train_model(
    resnet18, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=5, model_name="ResNet18-TL"
)

print("\nPhase 2: Fine-tuning (all layers)...")
for param in resnet18.parameters():
    param.requires_grad = True
optimizer = optim.Adam(resnet18.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

resnet18, hist_r18_ft, time_r18_ft = train_model(
    resnet18, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="ResNet18-FT"
)
time_resnet18 = time_r18_tl + time_r18_ft

## Step 10 — ResNet50 (Transfer Learning + Fine-tuning)

**ResNet50** (25.6M parameters) is a deeper variant of ResNet that uses bottleneck blocks (1x1 → 3x3 → 1x1 convolutions) for greater capacity with manageable computation. The same two-phase training strategy is applied:

- **Phase 1:** Frozen backbone, train classifier only (lr=1e-3, 5 epochs)
- **Phase 2:** Full fine-tuning (lr=5e-5, 10 epochs) — note the even lower learning rate to preserve the deeper network's pretrained features

We expect ResNet50 to slightly outperform ResNet18 due to increased depth, but at the cost of longer training time.

In [ ]:
print("=" * 60)
print("Training ResNet50 — Phase 1: Transfer Learning...")
print("=" * 60)

resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet50.fc = nn.Linear(resnet50.fc.in_features, NUM_CLASSES)

for param in resnet50.parameters():
    param.requires_grad = False
for param in resnet50.fc.parameters():
    param.requires_grad = True

resnet50 = resnet50.to(DEVICE)
optimizer = optim.Adam(resnet50.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

resnet50, hist_r50_tl, time_r50_tl = train_model(
    resnet50, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=5, model_name="ResNet50-TL"
)

print("\nPhase 2: Fine-tuning...")
for param in resnet50.parameters():
    param.requires_grad = True
optimizer = optim.Adam(resnet50.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

resnet50, hist_r50_ft, time_r50_ft = train_model(
    resnet50, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="ResNet50-FT"
)
time_resnet50 = time_r50_tl + time_r50_ft

## Step 11 — EfficientNet-B0 (Transfer Learning + Fine-tuning)

**EfficientNet-B0** (5.3M parameters) uses a compound scaling method that uniformly scales depth, width, and resolution. It achieves competitive accuracy with significantly fewer parameters than ResNet50, making it more efficient.

The same two-phase strategy is applied:
- **Phase 1:** Frozen features, train classifier head only (5 epochs)
- **Phase 2:** Full fine-tuning with lr=5e-5 (10 epochs)

EfficientNet's mobile-inverted bottleneck blocks (MBConv) with squeeze-and-excitation are particularly effective at capturing fine-grained features in clothing textures.

In [ ]:
print("=" * 60)
print("Training EfficientNet-B0 — Phase 1: Transfer Learning...")
print("=" * 60)

effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, NUM_CLASSES)

for param in effnet.parameters():
    param.requires_grad = False
for param in effnet.classifier.parameters():
    param.requires_grad = True

effnet = effnet.to(DEVICE)
optimizer = optim.Adam(effnet.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

effnet, hist_eff_tl, time_eff_tl = train_model(
    effnet, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=5, model_name="EfficientNet-B0-TL"
)

print("\nPhase 2: Fine-tuning...")
for param in effnet.parameters():
    param.requires_grad = True
optimizer = optim.Adam(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

effnet, hist_eff_ft, time_eff_ft = train_model(
    effnet, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="EfficientNet-B0-FT"
)
time_effnet = time_eff_tl + time_eff_ft

## Step 12 — VGG16 (Transfer Learning + Fine-tuning)

**VGG16** (138M parameters) is one of the classic deep CNN architectures, using a simple and uniform design of 3x3 convolution filters stacked in increasing depth (64 → 128 → 256 → 512). Despite its large parameter count (mostly in the fully-connected layers), VGG16 remains a strong baseline for transfer learning.

- **Phase 1:** Freeze all convolutional feature layers, train the 3-layer classifier head (5 epochs)
- **Phase 2:** Full fine-tuning with lr=5e-5 (10 epochs)

VGG16's simplicity makes it easy to interpret, but its large size results in slower training compared to more modern architectures like EfficientNet.

In [ ]:
print("=" * 60)
print("Training VGG16 — Phase 1: Transfer Learning...")
print("=" * 60)

vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
vgg16.classifier[6] = nn.Linear(vgg16.classifier[6].in_features, NUM_CLASSES)

for param in vgg16.features.parameters():
    param.requires_grad = False

vgg16 = vgg16.to(DEVICE)
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

vgg16, hist_vgg_tl, time_vgg_tl = train_model(
    vgg16, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=5, model_name="VGG16-TL"
)

print("\nPhase 2: Fine-tuning...")
for param in vgg16.parameters():
    param.requires_grad = True
optimizer = optim.Adam(vgg16.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

vgg16, hist_vgg_ft, time_vgg_ft = train_model(
    vgg16, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="VGG16-FT"
)
time_vgg16 = time_vgg_tl + time_vgg_ft

## Step 13 — DenseNet121 (Transfer Learning + Fine-tuning)

**DenseNet121** (8.0M parameters) introduces **dense connectivity** — each layer receives feature maps from all preceding layers. This encourages feature reuse, strengthens gradient flow, and reduces the number of parameters compared to ResNet.

- **Phase 1:** Freeze dense feature blocks, train classifier only (5 epochs)
- **Phase 2:** Full fine-tuning with lr=5e-5 (10 epochs)

DenseNet's dense connections are particularly useful for distinguishing subtle differences between visually similar classes (e.g., Shirt vs. T-shirt vs. Pullover).

In [ ]:
print("=" * 60)
print("Training DenseNet121 — Phase 1: Transfer Learning...")
print("=" * 60)

densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
densenet.classifier = nn.Linear(densenet.classifier.in_features, NUM_CLASSES)

for param in densenet.features.parameters():
    param.requires_grad = False

densenet = densenet.to(DEVICE)
optimizer = optim.Adam(densenet.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

densenet, hist_dn_tl, time_dn_tl = train_model(
    densenet, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=5, model_name="DenseNet121-TL"
)

print("\nPhase 2: Fine-tuning...")
for param in densenet.parameters():
    param.requires_grad = True
optimizer = optim.Adam(densenet.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

densenet, hist_dn_ft, time_dn_ft = train_model(
    densenet, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="DenseNet121-FT"
)
time_densenet = time_dn_tl + time_dn_ft

## Step 14 — Baseline Model Comparison

We evaluate all six trained models on the **held-out test set** using three key metrics:
- **Accuracy** — overall correctness
- **F1-Macro** — harmonic mean of precision and recall, averaged equally across all 10 classes (robust to class imbalance)
- **ROC-AUC** — area under the One-vs-Rest ROC curve (measures ranking quality of class probabilities)

We also record **training time** to assess the accuracy-vs-efficiency trade-off. This comparison reveals which architecture offers the best balance of performance and computational cost.

In [ ]:
print("\n" + "=" * 60)
print("  Baseline Model Comparison")
print("=" * 60)

baseline_results = {}
for name, model_obj, elapsed in [
    ("Baseline CNN",    baseline_cnn, time_baseline),
    ("ResNet18",        resnet18,     time_resnet18),
    ("ResNet50",        resnet50,     time_resnet50),
    ("EfficientNet-B0", effnet,       time_effnet),
    ("VGG16",           vgg16,        time_vgg16),
    ("DenseNet121",     densenet,     time_densenet),
]:
    _, acc, preds, trues, probs = evaluate(model_obj, test_loader, criterion, DEVICE)
    f1  = f1_score(trues, preds, average='macro')
    auc_val = roc_auc_score(label_binarize(trues, classes=range(NUM_CLASSES)),
                            probs, multi_class='ovr', average='macro')
    baseline_results[name] = {
        'accuracy': acc, 'f1': f1, 'auc': auc_val,
        'time': elapsed, 'preds': preds, 'trues': trues, 'probs': probs
    }
    print(f"  {name:20s} | Acc={acc:.4f} | F1={f1:.4f} | AUC={auc_val:.4f} | Time={elapsed:.1f}s")

## Step 15 — 5-Fold Cross-Validation on ResNet18

To obtain a **statistically robust** estimate of ResNet18's performance, we run 5-fold stratified cross-validation on the training set. Each fold:
1. Trains a fresh ResNet18 (pretrained weights) for 5 epochs on 4/5 of the training data
2. Evaluates F1-macro on the held-out 1/5 fold

The mean and standard deviation of F1 across folds indicate model **stability** — a low std confirms that performance is not highly sensitive to the particular train/val split. This is more reliable than a single train-test evaluation.

In [ ]:
print("\n" + "=" * 60)
print("  5-Fold Cross-Validation — ResNet18")
print("=" * 60)

fold_f1s = []
for fold_i, (fold_tr, fold_vl) in enumerate(fold_splits):
    actual_tr = train_idx[fold_tr]
    actual_vl = train_idx[fold_vl]

    fold_train_ds = FashionSubset(all_images, all_labels, actual_tr, transform_train)
    fold_val_ds   = FashionSubset(all_images, all_labels, actual_vl, transform_eval)
    fold_train_ld = DataLoader(fold_train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
    fold_val_ld   = DataLoader(fold_val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

    fold_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    fold_model.fc = nn.Linear(fold_model.fc.in_features, NUM_CLASSES)
    fold_model = fold_model.to(DEVICE)
    fold_opt = optim.Adam(fold_model.parameters(), lr=1e-4, weight_decay=1e-4)
    fold_sched = optim.lr_scheduler.ReduceLROnPlateau(fold_opt, patience=2, factor=0.5)

    fold_model, _, _ = train_model(
        fold_model, fold_train_ld, fold_val_ld, criterion, fold_opt, fold_sched,
        DEVICE, epochs=5, model_name=f"Fold-{fold_i+1}"
    )
    _, _, fp, ft, _ = evaluate(fold_model, fold_val_ld, criterion, DEVICE)
    fold_f1 = f1_score(ft, fp, average='macro')
    fold_f1s.append(fold_f1)
    print(f"  Fold {fold_i+1} F1-macro: {fold_f1:.4f}")
    del fold_model; torch.cuda.empty_cache()

print(f"\n  5-Fold CV F1-macro: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")

## Step 16 — Data Augmentation Impact Study (Traditional)

We study how different levels of **traditional augmentation** affect model performance by training three identical ResNet18 models with varying augmentation pipelines:

| Strategy | Transforms Applied |
|----------|-------------------|
| **No Augmentation** | Resize + Normalize only |
| **Traditional Aug** | Horizontal flip, rotation (+-10 deg), translation (10%) |
| **Strong Aug** | Flip, rotation (+-20 deg), translation (15%), scale (0.85-1.15), shear (10 deg), color jitter, perspective distortion, random erasing |

Augmentation creates artificial training variations that help the model learn **invariant features** and reduce overfitting. However, overly aggressive augmentation can distort class-defining features and hurt performance.

In [ ]:
print("\n" + "=" * 60)
print("  Data Augmentation Impact Study")
print("=" * 60)

# No augmentation
transform_no_aug = T.Compose([
    T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize([train_mean]*3, [train_std]*3),
])

# Strong augmentation
transform_strong_aug = T.Compose([
    T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=10),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomPerspective(distortion_scale=0.2, p=0.3),
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize([train_mean]*3, [train_std]*3),
    T.RandomErasing(p=0.2),
])

aug_results = {}
for aug_name, aug_transform in [
    ("No Augmentation", transform_no_aug),
    ("Traditional Aug", transform_train),
    ("Strong Aug",      transform_strong_aug),
]:
    print(f"\n  --- {aug_name} ---")
    ds_aug = FashionSubset(all_images, all_labels, train_idx, aug_transform)
    ld_aug = DataLoader(ds_aug, batch_size=BATCH, shuffle=True, num_workers=2)

    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m = m.to(DEVICE)
    opt = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)

    m, _, _ = train_model(m, ld_aug, val_loader, criterion, opt, sch,
                          DEVICE, epochs=7, model_name=aug_name)
    _, acc, preds, trues, probs = evaluate(m, test_loader, criterion, DEVICE)
    f1 = f1_score(trues, preds, average='macro')
    aug_results[aug_name] = {'accuracy': acc, 'f1': f1}
    print(f"  {aug_name}: Acc={acc:.4f} F1={f1:.4f}")
    del m; torch.cuda.empty_cache()

## Step 17 — Mixed-Based Augmentation (Mixup & CutMix)

Beyond traditional pixel-level transforms, we apply **sample-mixing augmentation** techniques:

- **Mixup** — creates new training samples by linearly interpolating two random images and their labels: `x_mix = lambda * x_a + (1 - lambda) * x_b`. This smooths decision boundaries and acts as a regularizer.
- **CutMix** — replaces a rectangular patch of one image with a patch from another, mixing the labels proportionally to the patch area. This forces the model to attend to all parts of the image rather than relying on a single discriminative region.

Both techniques use the Beta distribution to sample the mixing coefficient, and the loss is computed as a weighted combination of the losses for both labels. These methods have been shown to improve calibration and robustness in image classification.

In [ ]:
print("\n" + "=" * 60)
print("  Mixed-Based Augmentation (Mixup & CutMix)")
print("=" * 60)

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch_mix(model, loader, criterion, optimizer, device, mix_fn):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        mixed_imgs, y_a, y_b, lam = mix_fn(imgs, labels)
        optimizer.zero_grad()
        outputs = model(mixed_imgs)
        loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (lam * (outputs.argmax(1) == y_a).float()
                    + (1 - lam) * (outputs.argmax(1) == y_b).float()).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

for mix_name, mix_fn in [("Mixup", mixup_data), ("CutMix", cutmix_data)]:
    print(f"\n  --- {mix_name} ---")
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m = m.to(DEVICE)
    opt = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)

    for epoch in range(7):
        tr_loss, tr_acc = train_one_epoch_mix(m, train_loader, criterion, opt, DEVICE, mix_fn)
        vl_loss, vl_acc, vp, vt, _ = evaluate(m, val_loader, criterion, DEVICE)
        print(f"    Epoch {epoch+1}/7 | Train Loss={tr_loss:.4f} | Val Acc={vl_acc:.4f}")

    _, acc, preds, trues, probs = evaluate(m, test_loader, criterion, DEVICE)
    f1 = f1_score(trues, preds, average='macro')
    aug_results[mix_name] = {'accuracy': acc, 'f1': f1}
    print(f"  {mix_name}: Acc={acc:.4f} F1={f1:.4f}")
    del m; torch.cuda.empty_cache()

## Step 18 — Generative AI-Based Augmentation

We explore a **generative augmentation** approach inspired by diffusion models. Instead of training a full generative model (which would be computationally expensive), we simulate the denoising diffusion concept by injecting controlled **Gaussian noise** into the feature space:

- With 50% probability, each training image receives additive Gaussian noise (std=0.15) after the standard augmentation pipeline
- This mimics the idea of exploring nearby points on the data manifold, similar to how denoising diffusion models learn to generate images by progressively removing noise

This technique acts as a form of **implicit regularization**, encouraging the model to be robust to small perturbations and improving generalization to unseen variations.

In [ ]:
print("\n" + "=" * 60)
print("  Generative AI-Based Augmentation (Gaussian noise injection)")
print("=" * 60)

class GenerativeAugTransform:
    """Augmentation that adds Gaussian noise at multiple scales,
    simulating generative perturbations of the data manifold."""
    def __init__(self, base_transform, noise_std=0.15):
        self.base_transform = base_transform
        self.noise_std = noise_std

    def __call__(self, img):
        x = self.base_transform(img)
        if random.random() < 0.5:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise
        return x

gen_transform = GenerativeAugTransform(transform_train, noise_std=0.15)
gen_ds = FashionSubset(all_images, all_labels, train_idx, gen_transform)
gen_loader = DataLoader(gen_ds, batch_size=BATCH, shuffle=True, num_workers=2)

m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
m = m.to(DEVICE)
opt = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)
sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)

m, _, _ = train_model(m, gen_loader, val_loader, criterion, opt, sch,
                      DEVICE, epochs=7, model_name="GenAI Aug")
_, acc, preds, trues, probs = evaluate(m, test_loader, criterion, DEVICE)
f1 = f1_score(trues, preds, average='macro')
aug_results["GenAI Aug"] = {'accuracy': acc, 'f1': f1}
print(f"  GenAI Aug: Acc={acc:.4f} F1={f1:.4f}")
del m; torch.cuda.empty_cache()

## Step 19 — Augmentation Comparison Plot

We visualize the impact of all five augmentation strategies (No Augmentation, Traditional, Strong, Mixup, CutMix, and GenAI) side-by-side, comparing both Accuracy and F1-Macro scores. This plot helps identify which augmentation strategy offers the best trade-off for Fashion-MNIST classification.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
aug_names = list(aug_results.keys())
aug_accs  = [aug_results[n]['accuracy'] for n in aug_names]
aug_f1s   = [aug_results[n]['f1'] for n in aug_names]
x = np.arange(len(aug_names))
w = 0.35
ax.bar(x - w/2, aug_accs, w, label='Accuracy', color='#2E75B6', edgecolor='white')
ax.bar(x + w/2, aug_f1s,  w, label='F1-Macro', color='#70AD47', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(aug_names, rotation=20, ha='right')
ax.set_ylabel("Score"); ax.set_ylim(0.7, 1.0)
ax.set_title("Augmentation Strategy Comparison", fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i in range(len(aug_names)):
    ax.text(i - w/2, aug_accs[i] + 0.003, f"{aug_accs[i]:.3f}", ha='center', fontsize=7)
    ax.text(i + w/2, aug_f1s[i] + 0.003,  f"{aug_f1s[i]:.3f}",  ha='center', fontsize=7)
plt.tight_layout()
plt.savefig("fig2_augmentation_comparison.png", bbox_inches='tight')
plt.show()
print("Saved fig2_augmentation_comparison.png")

## Step 20 — Vision Transformer (ViT-Tiny)

We now move beyond CNNs to evaluate a **Vision Transformer (ViT)**. ViT-Tiny splits each 224x224 image into 16x16 patches (196 patches total), linearly embeds them, and processes the sequence through a standard Transformer encoder with self-attention.

Key architectural differences from CNNs:
- **No inductive bias** for locality — ViT learns spatial relationships purely from data via self-attention
- **Global receptive field** from layer 1 — every patch can attend to every other patch
- **Requires more data** to match CNNs, but pretrained ViTs transfer well to smaller datasets

We use `vit_tiny_patch16_224` from the **timm** library (pretrained on ImageNet-21k), fine-tuned with AdamW optimizer (lr=5e-5, weight decay=0.01) for 10 epochs.

In [ ]:
print("\n" + "=" * 60)
print("  Vision Transformer (ViT-Tiny)")
print("=" * 60)

vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
vit_model = vit_model.to(DEVICE)
optimizer = optim.AdamW(vit_model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

vit_model, hist_vit, time_vit = train_model(
    vit_model, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="ViT-Tiny"
)

_, vit_acc, vit_preds, vit_trues, vit_probs = evaluate(vit_model, test_loader, criterion, DEVICE)
vit_f1  = f1_score(vit_trues, vit_preds, average='macro')
vit_auc = roc_auc_score(label_binarize(vit_trues, classes=range(NUM_CLASSES)),
                         vit_probs, multi_class='ovr', average='macro')
print(f"  ViT-Tiny: Acc={vit_acc:.4f} | F1={vit_f1:.4f} | AUC={vit_auc:.4f} | Time={time_vit:.1f}s")
baseline_results["ViT-Tiny"] = {
    'accuracy': vit_acc, 'f1': vit_f1, 'auc': vit_auc,
    'time': time_vit, 'preds': vit_preds, 'trues': vit_trues, 'probs': vit_probs
}

## Step 21 — Hybrid CNN-Transformer Model

We design a custom **hybrid architecture** that combines the strengths of both CNNs and Transformers:

1. **CNN Backbone (EfficientNet-B0)** — extracts local spatial features, producing a 7x7 grid of 1280-dimensional feature maps
2. **Transformer Encoder (2 layers, 8 heads)** — treats the 49 spatial positions (7x7) as a sequence and applies multi-head self-attention to capture global relationships between features
3. **Global Average Pooling + Classifier** — averages across the sequence and maps to 10 classes

This hybrid approach leverages CNN's efficiency at local feature extraction with Transformer's ability to model long-range dependencies. The CNN reduces the sequence length (from 196 patches to 49 positions), making the Transformer computationally tractable.

In [ ]:
print("\n" + "=" * 60)
print("  Hybrid Model (EfficientNet backbone + Transformer head)")
print("=" * 60)

class HybridCNNTransformer(nn.Module):
    """CNN feature extractor -> Transformer encoder -> classifier."""
    def __init__(self, num_classes=10):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features   # (B, 1280, 7, 7)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=1280, nhead=8, dim_feedforward=512, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Sequential(
            nn.LayerNorm(1280),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        feat = self.features(x)
        B, C, H, W = feat.shape
        seq = feat.reshape(B, C, H * W).permute(0, 2, 1)  # (B, 49, 1280)
        seq = self.transformer(seq)
        cls_token = seq.mean(dim=1)
        return self.classifier(cls_token)

hybrid = HybridCNNTransformer(NUM_CLASSES).to(DEVICE)
optimizer = optim.AdamW(hybrid.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

hybrid, hist_hybrid, time_hybrid = train_model(
    hybrid, train_loader, val_loader, criterion, optimizer, scheduler,
    DEVICE, epochs=10, model_name="Hybrid CNN-Transformer"
)

_, hyb_acc, hyb_preds, hyb_trues, hyb_probs = evaluate(hybrid, test_loader, criterion, DEVICE)
hyb_f1  = f1_score(hyb_trues, hyb_preds, average='macro')
hyb_auc = roc_auc_score(label_binarize(hyb_trues, classes=range(NUM_CLASSES)),
                         hyb_probs, multi_class='ovr', average='macro')
print(f"  Hybrid: Acc={hyb_acc:.4f} | F1={hyb_f1:.4f} | AUC={hyb_auc:.4f} | Time={time_hybrid:.1f}s")
baseline_results["Hybrid CNN-TF"] = {
    'accuracy': hyb_acc, 'f1': hyb_f1, 'auc': hyb_auc,
    'time': time_hybrid, 'preds': hyb_preds, 'trues': hyb_trues, 'probs': hyb_probs
}

## Step 22 — Class Imbalance Handling: Focal Loss & Weighted Sampling

Although Fashion-MNIST is nearly balanced, we experiment with two class imbalance handling techniques to study their effect:

**Focal Loss (gamma=2):**
- Modifies cross-entropy by down-weighting well-classified examples: `FL = -(1-p_t)^gamma * log(p_t)`
- When gamma > 0, easy examples (high p_t) contribute less to the loss, focusing training on hard, misclassified samples
- Combined with per-class alpha weights (inverse frequency) to further emphasize underrepresented classes

**Weighted Random Sampling:**
- Instead of uniform random sampling during training, samples are drawn with probability inversely proportional to their class frequency
- This ensures the model sees each class equally often per epoch, regardless of the original class distribution
- Particularly effective for severely imbalanced datasets (not strictly necessary here, but demonstrates the technique)

Both techniques are evaluated with ResNet18 as the backbone for a fair comparison.

In [ ]:
print("\n" + "=" * 60)
print("  Class Imbalance Handling — Focal Loss & Weighted Sampling")
print("=" * 60)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

# Compute class weights
train_labels = all_labels[train_idx]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# Focal Loss
print("\n  --- Focal Loss (gamma=2) ---")
focal_criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)

m_focal = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m_focal.fc = nn.Linear(m_focal.fc.in_features, NUM_CLASSES)
m_focal = m_focal.to(DEVICE)
opt = optim.Adam(m_focal.parameters(), lr=1e-4, weight_decay=1e-4)
sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)

m_focal, _, _ = train_model(m_focal, train_loader, val_loader, focal_criterion, opt, sch,
                             DEVICE, epochs=7, model_name="Focal Loss")
_, focal_acc, fp, ft, fprobs = evaluate(m_focal, test_loader, criterion, DEVICE)
focal_f1 = f1_score(ft, fp, average='macro')
print(f"  Focal Loss: Acc={focal_acc:.4f} F1={focal_f1:.4f}")
del m_focal; torch.cuda.empty_cache()

# Weighted Sampling
print("\n  --- Weighted Random Sampling ---")
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
weighted_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2)

m_ws = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m_ws.fc = nn.Linear(m_ws.fc.in_features, NUM_CLASSES)
m_ws = m_ws.to(DEVICE)
opt = optim.Adam(m_ws.parameters(), lr=1e-4, weight_decay=1e-4)
sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)

m_ws, _, _ = train_model(m_ws, weighted_loader, val_loader, criterion, opt, sch,
                          DEVICE, epochs=7, model_name="Weighted Sampling")
_, ws_acc, wp, wt, wprobs = evaluate(m_ws, test_loader, criterion, DEVICE)
ws_f1 = f1_score(wt, wp, average='macro')
print(f"  Weighted Sampling: Acc={ws_acc:.4f} F1={ws_f1:.4f}")
del m_ws; torch.cuda.empty_cache()

## Step 23 — Explainability: Grad-CAM

**Grad-CAM (Gradient-weighted Class Activation Mapping)** provides visual explanations for CNN predictions by:
1. Computing gradients of the target class score with respect to the final convolutional layer's feature maps
2. Weighting each feature map by its gradient (global average pooled) to produce a heatmap
3. Overlaying the heatmap on the original image to show which regions the model focuses on

We apply Grad-CAM to ResNet18's `layer4` (the last residual block) for one sample from each of the 10 classes. This reveals whether the model attends to semantically meaningful regions (e.g., shoe soles, bag straps, collar shapes) or relies on spurious correlations.

In [ ]:
print("\n" + "=" * 60)
print("  Grad-CAM Explainability (ResNet18)")
print("=" * 60)

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from skimage.transform import resize as sk_resize

target_layers = [resnet18.layer4[-1]]
cam = GradCAM(model=resnet18, target_layers=target_layers)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Grad-CAM Visualizations (ResNet18)", fontsize=14, fontweight='bold')

for i in range(10):
    ax = axes[i // 5, i % 5]
    class_mask = np.where(all_labels[test_idx] == i)[0]
    sample_idx_gc = class_mask[0]
    img_raw = all_images[test_idx[sample_idx_gc]]

    inp = transform_eval(img_raw).unsqueeze(0).to(DEVICE)
    targets = [ClassifierOutputTarget(i)]
    grayscale_cam = cam(input_tensor=inp, targets=targets)[0]

    img_rgb = np.stack([img_raw / 255.0] * 3, axis=-1).astype(np.float32)
    img_rgb_resized = sk_resize(img_rgb, (IMG_SIZE, IMG_SIZE, 3), anti_aliasing=True)
    visualization = show_cam_on_image(img_rgb_resized, grayscale_cam, use_rgb=True)

    ax.imshow(visualization)
    ax.set_title(CLASS_NAMES[i], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig("fig3_gradcam.png", bbox_inches='tight')
plt.show()
print("Saved fig3_gradcam.png")

## Step 24 — Explainability: SHAP (Deep Explainer)

**SHAP (SHapley Additive exPlanations)** provides a game-theoretic approach to feature attribution. Unlike Grad-CAM which only highlights spatial regions, SHAP computes the **exact contribution of each pixel** to the model's prediction:

- **Background set:** One representative image per class (10 images) used as the reference distribution
- **Test set:** 5 test images (one per class) for which we compute SHAP values
- **DeepExplainer:** Approximates SHAP values using DeepLIFT's backpropagation rules, which is faster than exact Shapley computation

The resulting heatmaps show:
- **Red pixels** — positively contribute to the predicted class (increase confidence)
- **Blue pixels** — negatively contribute (decrease confidence)

This complements Grad-CAM by providing pixel-level, class-specific attribution rather than coarse spatial heatmaps.

In [ ]:
print("\n" + "=" * 60)
print("  SHAP Explainability (ResNet18)")
print("=" * 60)

import shap

resnet18.eval()

# Prepare background and test samples
bg_indices = [np.where(all_labels[train_idx] == c)[0][0] for c in range(NUM_CLASSES)]
background = torch.stack([transform_eval(all_images[train_idx[i]]) for i in bg_indices]).to(DEVICE)

test_indices = [np.where(all_labels[test_idx] == c)[0][0] for c in range(5)]  # 5 samples
test_imgs = torch.stack([transform_eval(all_images[test_idx[i]]) for i in test_indices]).to(DEVICE)

explainer = shap.DeepExplainer(resnet18, background)
shap_values = explainer.shap_values(test_imgs)

# Visualize SHAP for channel 0 (grayscale repeated across 3 channels)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("SHAP Feature Attribution (ResNet18)", fontsize=14, fontweight='bold')

for i in range(5):
    # Original image
    ax = axes[0, i]
    img_raw = all_images[test_idx[test_indices[i]]]
    ax.imshow(img_raw, cmap='gray')
    with torch.no_grad():
        pred_class = resnet18(test_imgs[i:i+1]).argmax(1).item()
    ax.set_title(f"True: {CLASS_NAMES[all_labels[test_idx[test_indices[i]]]]}", fontsize=8)
    ax.axis('off')

    # SHAP values (sum across channels)
    ax = axes[1, i]
    sv = shap_values[pred_class] if isinstance(shap_values, list) else shap_values[..., pred_class]
    shap_img = np.array(sv[i]).sum(axis=0)  # sum over channels
    vmax = np.abs(shap_img).max()
    ax.imshow(shap_img, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f"Pred: {CLASS_NAMES[pred_class]}", fontsize=8)
    ax.axis('off')

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("SHAP values", fontsize=11)

plt.tight_layout()
plt.savefig("fig3b_shap.png", bbox_inches='tight')
plt.show()
print("Saved fig3b_shap.png")

## Step 25 — Confusion Matrices (All Models)

Normalized confusion matrices for all 8 models reveal **per-class classification patterns**:
- Diagonal values close to 1.0 indicate strong performance for that class
- Off-diagonal values highlight systematic confusions between similar classes

Common confusion patterns in Fashion-MNIST:
- **Shirt ↔ T-shirt/top ↔ Pullover ↔ Coat** — these upper-body garments share similar silhouettes
- **Sneaker ↔ Ankle boot** — footwear classes with overlapping features
- **Sandal, Bag, Trouser** — typically well-separated due to distinct visual features

In [ ]:
short_cls = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
             'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

plot_models = list(baseline_results.items())
n_models = len(plot_models)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
fig.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight='bold', y=1.01)
axes = axes.flatten()

for idx, (name, res) in enumerate(plot_models):
    ax = axes[idx]
    cm = confusion_matrix(res['trues'], res['preds'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(short_cls, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(short_cls, fontsize=7)
    ax.set_title(f"{name}\nAcc={res['accuracy']:.3f} F1={res['f1']:.3f}",
                 fontweight='bold', fontsize=10)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha='center', va='center',
                    fontsize=6, color='white' if cm_norm[i,j] > 0.5 else 'black')

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("fig4_confusion_matrices.png", bbox_inches='tight')
plt.show()
print("Saved fig4_confusion_matrices.png")

## Step 26 — ROC Curves (One-vs-Rest)

**ROC (Receiver Operating Characteristic) curves** plot the True Positive Rate (TPR) against the False Positive Rate (FPR) at various classification thresholds for each class in a One-vs-Rest setting.

- **AUC = 1.0** — perfect classifier that separates the class from all others
- **AUC = 0.5** — random classifier (diagonal line)

Per-class AUC values reveal which classes are hardest to discriminate. Classes with high visual similarity to others (e.g., Shirt) will have lower per-class AUC, while distinctive classes (e.g., Bag, Trouser) will approach 1.0. The macro-averaged AUC provides an overall summary of the model's ranking quality.

In [ ]:
colors_roc = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
fig.suptitle("ROC Curves (One-vs-Rest)", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (name, res) in enumerate(plot_models):
    ax = axes[idx]
    y_bin = label_binarize(res['trues'], classes=range(NUM_CLASSES))
    for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors_roc)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], res['probs'][:, i])
        ax.plot(fpr, tpr, color=col, lw=1.2,
                label=f"{short_cls[i]} ({auc(fpr, tpr):.3f})")
    ax.plot([0,1],[0,1], 'k--', lw=0.7, alpha=0.4)
    ax.set_title(f"{name} (AUC={res['auc']:.3f})", fontweight='bold', fontsize=10)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=6, loc='lower right'); ax.grid(alpha=0.2)

for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("fig5_roc_curves.png", bbox_inches='tight')
plt.show()
print("Saved fig5_roc_curves.png")

## Step 27 — Precision-Recall Curves

**Precision-Recall (PR) curves** are especially informative for evaluating per-class performance:
- **Precision** = TP / (TP + FP) — of all predicted positives, how many are correct?
- **Recall** = TP / (TP + FN) — of all actual positives, how many are found?

The **Average Precision (AP)** summarizes the PR curve as the area under it. PR curves are preferred over ROC curves when classes are imbalanced, as they do not get inflated by a large number of true negatives.

The macro-averaged mAP gives a single score for overall model quality across all classes.

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
fig.suptitle("Precision-Recall Curves (One-vs-Rest)", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (name, res) in enumerate(plot_models):
    ax = axes[idx]
    y_bin = label_binarize(res['trues'], classes=range(NUM_CLASSES))
    macro_ap = average_precision_score(y_bin, res['probs'], average='macro')
    for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors_roc)):
        prec, rec, _ = precision_recall_curve(y_bin[:, i], res['probs'][:, i])
        ap = average_precision_score(y_bin[:, i], res['probs'][:, i])
        ax.plot(rec, prec, color=col, lw=1.2,
                label=f"{short_cls[i]} ({ap:.3f})")
    ax.set_title(f"{name} (mAP={macro_ap:.3f})", fontweight='bold', fontsize=10)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(fontsize=6, loc='upper right'); ax.grid(alpha=0.2)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("fig6_pr_auc.png", bbox_inches='tight')
plt.show()
print("Saved fig6_pr_auc.png")

## Step 28 — t-SNE Visualization of Learned Embeddings

**t-SNE (t-distributed Stochastic Neighbor Embedding)** projects high-dimensional feature representations (512-dim from ResNet18's penultimate layer) into 2D for visualization.

We extract embeddings from ResNet18's `avgpool` layer for 2,000 test samples using a forward hook. Well-separated clusters in the t-SNE plot confirm that the model has learned **discriminative feature representations**:
- Tight, distinct clusters indicate the model can easily separate those classes
- Overlapping clusters reveal classes that the model finds hard to distinguish (expected for Shirt/T-shirt/Pullover)

t-SNE parameters: perplexity=30, 1000 iterations, PCA initialization for stability.

In [ ]:
print("Extracting ResNet18 embeddings for t-SNE...")

embeddings_list = []
def hook_fn(module, input, output):
    embeddings_list.append(output.detach().cpu().numpy())

hook = resnet18.avgpool.register_forward_hook(hook_fn)

n_tsne = 2000
tsne_subset = Subset(test_ds, range(min(n_tsne, len(test_ds))))
tsne_loader = DataLoader(tsne_subset, batch_size=64, shuffle=False)

tsne_labels = []
resnet18.eval()
with torch.no_grad():
    for imgs, labels in tsne_loader:
        _ = resnet18(imgs.to(DEVICE))
        tsne_labels.extend(labels.numpy())

hook.remove()
tsne_embeds = np.concatenate(embeddings_list, axis=0).squeeze()
tsne_labels = np.array(tsne_labels[:len(tsne_embeds)])

print(f"Running t-SNE on {len(tsne_embeds)} samples...")
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=SEED, init='pca')
emb_2d = tsne.fit_transform(tsne_embeds)

palette = sns.color_palette("tab10", NUM_CLASSES)
fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(CLASS_NAMES):
    mask = tsne_labels == i
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               label=cls, c=[palette[i]], alpha=0.6, s=15, edgecolors='none')
ax.set_title("t-SNE of ResNet18 Embeddings (Test Set)", fontsize=13, fontweight='bold')
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.legend(fontsize=8, loc='best', framealpha=0.8, markerscale=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("fig7_tsne.png", bbox_inches='tight')
plt.show()
print("Saved fig7_tsne.png")

## Step 29 — Model Performance Comparison Chart

A comprehensive visual comparison of all 8 models across two dimensions:

1. **Performance metrics (left):** Accuracy, F1-Macro, and ROC-AUC side-by-side — pretrained models should consistently outperform the baseline CNN, and the relative ranking reveals the best architecture for this task
2. **Training time (right):** Computational cost measured in seconds — VGG16 is expected to be the slowest (138M params), while EfficientNet-B0 offers the best performance-per-second ratio

This trade-off analysis is critical for practical deployment: a model that is 1% more accurate but 3x slower may not be worth the cost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight='bold')

model_names = list(baseline_results.keys())
accs  = [baseline_results[n]['accuracy'] for n in model_names]
f1s   = [baseline_results[n]['f1'] for n in model_names]
aucs  = [baseline_results[n]['auc'] for n in model_names]
times = [baseline_results[n]['time'] for n in model_names]

x = np.arange(len(model_names))
w = 0.25

ax = axes[0]
ax.bar(x - w, accs, w, label='Accuracy', color='#1F4E79', edgecolor='white')
ax.bar(x,     f1s,  w, label='F1-Macro', color='#2E75B6', edgecolor='white')
ax.bar(x + w, aucs, w, label='ROC-AUC',  color='#70AD47', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=25, ha='right')
ax.set_ylabel("Score"); ax.set_ylim(0.7, 1.02)
ax.set_title("Accuracy / F1 / AUC"); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for i in range(len(model_names)):
    for j, vals in enumerate([accs, f1s, aucs]):
        ax.text(i + (j-1)*w, vals[i] + 0.003, f"{vals[i]:.3f}", ha='center', fontsize=6)

ax = axes[1]
bars = ax.bar(model_names, times,
              color=sns.color_palette("Blues_d", len(model_names)), edgecolor='white')
ax.set_title("Training Time (seconds)"); ax.set_ylabel("Seconds")
ax.tick_params(axis='x', rotation=25)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{t:.0f}s", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("fig8_model_comparison.png", bbox_inches='tight')
plt.show()
print("Saved fig8_model_comparison.png")

## Step 30 — Threshold Analysis: Recall vs. False Positive Rate Trade-off

We analyze the **operational trade-off between Recall and False Positive Rate** for the hardest class — **Shirt** (class 6), which is frequently confused with T-shirt, Pullover, and Coat.

By varying the classification threshold from 0.05 to 0.95, we observe:
- **Low threshold (0.10-0.30):** High recall (catches most shirts) but more false positives
- **Default threshold (0.50):** Balanced precision and recall
- **High threshold (0.70+):** Very precise but misses many shirts (low recall)

The **recommended threshold of 0.30** maximizes the F1-score for Shirt detection, accepting a modest increase in false positives to catch significantly more true positives. This is analogous to a real-world scenario where missing a category is more costly than a false alarm (e.g., in product recommendation or inventory management).

In [ ]:
# Shirt (class 6) is the hardest class — often confused with T-shirt, Pullover, Coat
HARD_CLASS = 6
best_res = baseline_results["ResNet18"]
y_binary = (best_res['trues'] == HARD_CLASS).astype(int)
prob_hard = best_res['probs'][:, HARD_CLASS]

thresholds = np.arange(0.05, 0.95, 0.05)
recalls, precisions, f1s, fprs = [], [], [], []

for thresh in thresholds:
    y_pred_t = (prob_hard >= thresh).astype(int)
    tp = np.sum((y_pred_t == 1) & (y_binary == 1))
    fp = np.sum((y_pred_t == 1) & (y_binary == 0))
    fn = np.sum((y_pred_t == 0) & (y_binary == 1))
    tn = np.sum((y_pred_t == 0) & (y_binary == 0))
    recalls.append(tp / (tp + fn + 1e-9))
    precisions.append(tp / (tp + fp + 1e-9))
    f1s.append(2 * tp / (2 * tp + fp + fn + 1e-9))
    fprs.append(fp / (fp + tn + 1e-9))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Threshold Analysis — '{CLASS_NAMES[HARD_CLASS]}' Detection (ResNet18)",
             fontsize=13, fontweight='bold')

axes[0].plot(thresholds, recalls,    'b-o', ms=4, lw=2, label='Recall')
axes[0].plot(thresholds, precisions, 'g-s', ms=4, lw=2, label='Precision')
axes[0].plot(thresholds, f1s,        'r-^', ms=4, lw=2, label='F1-Score')
axes[0].axvline(0.30, color='orange', ls='--', lw=1.5, label='Recommended threshold=0.30')
axes[0].axvline(0.50, color='gray',   ls=':',  lw=1.0, label='Default threshold=0.50')
axes[0].set_xlabel("Classification Threshold"); axes[0].set_ylabel("Score")
axes[0].set_title("Recall / Precision / F1 vs Threshold")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(fprs, recalls, 'purple', lw=2, marker='o', ms=4)
for thresh, fpr, rec in zip(thresholds, fprs, recalls):
    if abs(thresh - 0.3) < 0.03 or abs(thresh - 0.5) < 0.03:
        axes[1].annotate(f'{thresh:.2f}', (fpr, rec),
                         textcoords='offset points', xytext=(8, -8), fontsize=9)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("Recall")
axes[1].set_title("Recall vs FPR Trade-off"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig9_threshold_analysis.png", bbox_inches='tight')
plt.show()
print("Saved fig9_threshold_analysis.png")

## Step 31 — Final Results Summary & Conclusions

### Summary of Findings

**1. Model Architecture Comparison:**
- All pretrained models significantly outperform the baseline CNN, confirming the value of **transfer learning** for small-to-medium datasets
- The two-phase training strategy (frozen backbone → full fine-tuning) consistently yields strong results
- **EfficientNet-B0** and **DenseNet121** typically offer the best accuracy-to-parameter ratio
- **ViT-Tiny** demonstrates that Vision Transformers can match CNN performance even on small grayscale images when pretrained on large-scale data
- The **Hybrid CNN-Transformer** combines local (CNN) and global (Transformer) feature processing for competitive results

**2. Data Augmentation Impact:**
- Traditional augmentation (flip, rotation, translation) provides consistent improvement over no augmentation
- Mixup and CutMix act as effective regularizers, particularly for reducing overconfident predictions
- Generative AI-based noise augmentation offers comparable benefits to traditional methods

**3. Explainability:**
- **Grad-CAM** confirms that models focus on semantically relevant regions (garment shapes, textures)
- **SHAP** provides finer-grained pixel-level attribution, useful for debugging and trust-building

**4. Class Imbalance Handling:**
- Focal Loss and Weighted Sampling show marginal impact on Fashion-MNIST (already balanced)
- These techniques would be more impactful on truly imbalanced datasets (e.g., medical imaging)

**5. Threshold Analysis:**
- Lowering the classification threshold for hard classes (Shirt) can substantially improve recall at the cost of modest precision loss
- This trade-off is application-dependent and should be tuned based on business requirements

In [ ]:
print("\n" + "=" * 70)
print("  FINAL RESULTS SUMMARY")
print("=" * 70)

summary = []
for name, res in baseline_results.items():
    prec = precision_score(res['trues'], res['preds'], average='macro', zero_division=0)
    rec  = recall_score(res['trues'], res['preds'], average='macro', zero_division=0)
    summary.append({
        'Model': name,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{prec:.4f}",
        'Recall': f"{rec:.4f}",
        'F1-Macro': f"{res['f1']:.4f}",
        'ROC-AUC': f"{res['auc']:.4f}",
        'Time (s)': f"{res['time']:.1f}",
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

summary_df.to_csv("results_summary.csv", index=False)
print("\nSaved results_summary.csv")

print("\nAll figures generated:")
for f in ["fig1_eda.png", "fig2_augmentation_comparison.png", "fig3_gradcam.png",
          "fig3b_shap.png", "fig4_confusion_matrices.png", "fig5_roc_curves.png",
          "fig6_pr_auc.png", "fig7_tsne.png", "fig8_model_comparison.png",
          "fig9_threshold_analysis.png", "results_summary.csv"]:
    print(f"  - {f}")

print(f"\n5-Fold CV F1 (ResNet18): {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")
print("\nAugmentation impact:")
for name, res in aug_results.items():
    print(f"  {name:20s}: Acc={res['accuracy']:.4f} F1={res['f1']:.4f}")
print(f"\nImbalance handling:")
print(f"  Focal Loss:        Acc={focal_acc:.4f} F1={focal_f1:.4f}")
print(f"  Weighted Sampling: Acc={ws_acc:.4f} F1={ws_f1:.4f}")

## Step 32 — Generate Word Report

Generate the final `.docx` report with all figures and results table embedded.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, 'generate_report.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print('Report PW8_Report.docx generated successfully!')